# 12 · TTE: модели по объектам vs глобальная модель

**Цель** — проверить, что выигрывает: одна глобальная discrete-hazard модель
и/или отдельная модель на **каждом объекте**. Модели обучаются так же, как в
`11_wear_tte` (person-time, цензура, holdout ≥ 2025-07-01), но локальная —
только на строках своего объекта.

- Обучение: `research/tte_per_object.py` (модели → `research/models/objects/`).
- Результаты: `models/per_object_results.csv` + `models/per_object_summary.json`.
- Метрики на holdout: Uno-C, meanAUC, IBS, PR-AUC(P(24ч)) — локальная и
  глобальная на **одних и тех же** holdout-субъектах объекта.

In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE

df = pd.read_csv(RESEARCH / "models" / "per_object_results.csv", dtype={"объект": str})
with open(RESEARCH / "models" / "per_object_summary.json", encoding="utf-8") as f:
    summ = json.load(f)

print("объектов:", len(df))
print("параметры:", {k: v for k, v in summ["params"].items()
                     if k in ("iters", "lr", "depth", "cap", "min_train")})

In [ ]:
# Таблица результатов по объектам (локальная vs глобальная на одних holdout-субъектах)
cols = ["объект", "n_train", "n_holdout", "ho_событий",
        "uno_c_лок", "uno_c_глоб", "auc_лок", "auc_глоб",
        "ibs_лок", "ibs_глоб", "pr24_лок", "pr24_глоб"]
df[cols].round(3).to_string(index=False)

In [ ]:
# Парные дельты и сводка
d_uno = df["uno_c_лок"] - df["uno_c_глоб"]
d_ibs = df["ibs_лок"] - df["ibs_глоб"]

print("Uno-C: лок лучше в %.0f%% объектов | mean=%+.3f med=%+.3f" % (
    100 * summ["deltas"]["uno_c_share_local_better"],
    summ["deltas"]["uno_c_mean"], summ["deltas"]["uno_c_med"]))
print("IBS:   лок лучше в %.0f%% объектов | mean=%+.3f med=%+.3f" % (
    100 * summ["deltas"]["ibs_share_local_better"],
    summ["deltas"]["ibs_mean"], summ["deltas"]["ibs_med"]))

winners = df.loc[d_uno > 0]
w = winners.assign(delta=(winners["uno_c_лок"] - winners["uno_c_глоб"]).round(3))
print("\nОбъекты, где локальная заметно лучше (delta Uno-C > +0.01):")
print(w.loc[d_uno[winners.index] > 0.01, ["объект", "n_train", "uno_c_лок", "uno_c_глоб", "delta"]]
      .to_string(index=False))

In [ ]:
# Графики сравнения: scatter лок vs глоб (Uno-C), дельты, IBS
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.scatter(df["uno_c_глоб"], df["uno_c_лок"], s=50, color="C0")
for _, r in df.iterrows():
    ax.annotate(int(r["объект"]), (r["uno_c_глоб"], r["uno_c_лок"]), fontsize=7)
ax.set_xlabel("глобальная (Uno-C)"); ax.set_ylabel("локальная (Uno-C)")
ax.set_title("Uno-C по объектам: локальная vs глобальная")
ax.grid(alpha=0.3)

ax = axes[1]
delta = (df["uno_c_лок"] - df["uno_c_глоб"]).sort_values()
ax.barh([str(o) for o in df.loc[delta.index, "объект"]], delta.values, color=[
    "C2" if v > 0 else "C3" for v in delta])
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("delta Uno-C (лок - глоб)")
ax.set_title("Дельта по объектам")

ax = axes[2]
ax.scatter(df["ibs_глоб"], df["ibs_лок"], s=50, color="C1")
ax.plot([0, 0.2], [0, 0.2], "--", color="gray")
ax.set_xlabel("глобальная (IBS)"); ax.set_ylabel("локальная (IBS)")
ax.set_title("IBS: локальная vs глобальная (ниже = лучше)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Итог по объединённой holdout-выборке (локальные по своим объектам vs глобальная)
print("Параметры обучения локальных: iters=%s cap=%s" % (summ["params"]["iters"], summ["params"]["cap"]))
print("Число объектов в эксперименте:", len(df))
print("Объединённая выборка holdout: ЛОКАЛЬНЫЕ vs ГЛОБАЛЬНАЯ")
print(pd.DataFrame({
    "Uno-C": [summ["union"]["local"]["uno_c"], summ["union"]["global"]["uno_c"]],
    "meanAUC": [summ["union"]["local"]["mean_auc"], summ["union"]["global"]["mean_auc"]],
    "IBS": [summ["union"]["local"]["ibs"], summ["union"]["global"]["ibs"]],
}, index=["локальные", "глобальная"]).round(3).T.to_string())

print("\nПобедитель (на объединённой выборке):",
      "глобальная" if summ["union"]["global"]["uno_c"] >= summ["union"]["local"]["uno_c"]
      else "локальные")

## Выводы

- Глобальная модель стабильна: на объединённой holdout-выборке она чуть лучше
  локальных (Uno-C 0.770 vs 0.764; IBS 0.104 vs 0.107).
- Локальные модели побеждают на **≈половине объектов** и существенно на
  объектах с собственной «сигнатурой» (напр. 111: 0.817 vs 0.768; 4369, 5218,
  5675, 5578 — выигрыш delta Uno-C > +0.02).
- Практический вывод — **гибридная схема**: для объекта использовать модель,
  выбранную по валидации (локальную, если она лучше на внутренней валидации,
  иначе глобальную), либо простой ансамбль локальная/глобальная.
- Объекты без истории до 2025-07 (например, `5963`) — cold-start: всегда
  глобальная модель, локальную со временем дообучать.
- Модели сохранены: `research/models/objects/tte_obj_<ид>.cbm`; метрики в
  `models/per_object_results.csv`, сводка в `models/per_object_summary.json`.